In [2]:
import os
import sys
from pathlib import Path
import re
import time
import random
import requests
import pandas as pd
import logging
from datetime import datetime, timedelta
from bs4 import BeautifulSoup
import json
from urllib.parse import quote

# auto-reload modules so don't have to restart the kernel when making changes
%load_ext autoreload
%autoreload 2

# Import settings from config
from config import (
    BASE_DIR, DATA_DIR, LOGS_DIR, TRANSCRIPTS_DIR, RAW_DIR,
    USER_AGENT, COMPANIES, TRANSCRIPT_METADATA
)

# Import all the functions we need from our collection module
from earnings_call_analyzer.transcript_collection import (
    BASE_URL, USER_AGENTS,
    get_random_user_agent, get_scraping_headers, wait_with_jitter,
    get_company_ir_transcript, get_seeking_alpha_transcript, get_yahoo_finance_transcript,
    process_company, update_metadata
)

# Settings for our collector
DELAY_BETWEEN_COMPANIES = 300  # 5 minute pause between companies to avoid getting blocked
MAX_QUARTERS_TO_CHECK = 8     # Look back at most 2 years (8 quarters)

# Set up logging to both file and console
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(os.path.join(LOGS_DIR, "transcript_collector.log")),
        logging.StreamHandler(sys.stdout)  # Also print to console
    ]
)

# Get a logger instance
logger = logging.getLogger(__name__)

# Helper function to print nice status messages
def status_print(message):
    """Prints status messages with timestamps for better tracking"""
    timestamp = datetime.now().strftime('%H:%M:%S')
    print(f"[{timestamp}] {message}")

# Direct links to each company's investor relations page
IR_URLS = {
    "AAPL": "https://investor.apple.com/investor-relations/default.aspx",
    "MSFT": "https://www.microsoft.com/en-us/investor/",
    "GOOGL": "https://abc.xyz/investor/",
    "AMZN": "https://ir.aboutamazon.com/quarterly-results",
    "META": "https://investor.fb.com/home/default.aspx",
    "TSLA": "https://ir.tesla.com/events",
    "NVDA": "https://investor.nvidia.com/events-and-presentations/events-and-presentations/default.aspx",
    "NFLX": "https://ir.netflix.net/investor-relations/overview/default.aspx",
    "JPM": "https://www.jpmorganchase.com/ir",
    "BAC": "https://investor.bankofamerica.com/",
    "WFC": "https://www.wellsfargo.com/about/investor-relations/",
    "GS": "https://www.goldmansachs.com/investor-relations/",
    "V": "https://investor.visa.com/",
    "WMT": "https://corporate.walmart.com/investors",
    "HD": "https://ir.homedepot.com/",
    "PG": "https://www.pginvestor.com/",
    "KO": "https://investors.coca-colacompany.com/",
    "PEP": "https://www.pepsico.com/investors",
    "JNJ": "https://www.investor.jnj.com/",
    "PFE": "https://investors.pfizer.com/",
    "UNH": "https://www.unitedhealthgroup.com/investors.html",
    "MRK": "https://www.merck.com/investor-relations/",
    "VZ": "https://www.verizon.com/about/investors",
    "T": "https://investors.att.com/",
}

# Wait a random amount of time to look more like a human
def verbose_wait(base_seconds=10):
    """Waits a bit with some random jitter to avoid looking like a bot"""
    jitter = random.uniform(0, 5)
    wait_time = base_seconds + jitter
    status_print(f"⏱ Waiting {wait_time:.2f}s (jitter: {jitter:.2f}s)...")
    time.sleep(wait_time)
    return wait_time

# Wrapper to make processing a company more informative in the console
def verbose_process_company(company, output_dir, ir_urls):
    """Process a company with nice console output to track progress"""
    ticker = company['ticker']
    status_print(f"🔍 Starting to process {ticker} ({company['name']})...")

    # Figure out which quarters we're looking for
    today = datetime.now()
    first_quarter = today
    last_quarter = today - timedelta(days=90*(MAX_QUARTERS_TO_CHECK-1))

    # Format the quarter range nicely
    first_q_num = (first_quarter.month-1)//3 + 1
    first_q_str = f"Q{first_q_num} {first_quarter.year}"

    last_q_num = (last_quarter.month-1)//3 + 1
    last_q_str = f"Q{last_q_num} {last_quarter.year}"

    status_print(f"  📅 Checking for {ticker} earnings from {first_q_str} back to {last_q_str} ({MAX_QUARTERS_TO_CHECK} quarters)")

    # Do the actual processing
    transcripts = process_company(company, output_dir, ir_urls)

    # Let us know what we found
    if transcripts:
        status_print(f"✅ Found {len(transcripts)} transcripts for {ticker}:")
        for t in transcripts:
            status_print(f"  📄 {t['quarter']} from {t['source']} ({t['transcript_length']} chars)")
    else:
        status_print(f"❌ No transcripts found for {ticker}")

    return transcripts

def main():
    """Main function that runs the whole transcript collection process"""
    status_print("🚀 STARTING EARNINGS CALL TRANSCRIPT COLLECTOR 🚀")
    status_print(f"Output directory: {TRANSCRIPTS_DIR}")

    all_transcripts = []

    # How many companies to process (all by default)
    num_companies_to_process = len(COMPANIES)
    # For testing, you can just process a few
    # num_companies_to_process = 2

    companies_to_process = COMPANIES[:num_companies_to_process]

    status_print(f"Will process {len(companies_to_process)} companies")

    # Show which companies we'll be processing
    for i, company in enumerate(companies_to_process):
        status_print(f"{i+1}. {company['ticker']} - {company['name']}")

    start_time = datetime.now()
    status_print(f"Start time: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

    for i, company in enumerate(companies_to_process):
        try:
            status_print(f"\n📊 COMPANY {i+1}/{len(companies_to_process)}: {company['ticker']} - {company['name']}")

            # Process this company
            transcripts = verbose_process_company(
                company, 
                output_dir=TRANSCRIPTS_DIR, 
                ir_urls=IR_URLS
            )
            all_transcripts.extend(transcripts)

            # Wait a while between companies to avoid getting blocked
            # (but don't wait after the last company)
            if i < len(companies_to_process) - 1:
                next_company = companies_to_process[i+1]
                status_print(f"⏳ Waiting {DELAY_BETWEEN_COMPANIES} seconds before processing {next_company['ticker']}...")

                # For long waits, show a countdown so we know it's not frozen
                if DELAY_BETWEEN_COMPANIES > 60:
                    for remaining in range(DELAY_BETWEEN_COMPANIES, 0, -60):
                        minutes_left = remaining // 60
                        if minutes_left > 0:
                            time.sleep(60)
                            status_print(f"  ⌛ {minutes_left} minutes remaining...")
                    time.sleep(remaining % 60)
                else:
                    time.sleep(DELAY_BETWEEN_COMPANIES)

                status_print("✅ Wait complete, continuing to next company.")

        except Exception as e:
            status_print(f"❌ ERROR: Failed to process {company['ticker']}: {str(e)}")
            logger.error(f"Error processing {company['ticker']}: {str(e)}")

    # Show a summary of what we found
    status_print("\n📋 COLLECTION SUMMARY:")
    if all_transcripts:
        # Count how many transcripts per company
        company_counts = {}
        for t in all_transcripts:
            ticker = t['ticker']
            company_counts[ticker] = company_counts.get(ticker, 0) + 1

        status_print(f"Total transcripts collected: {len(all_transcripts)}")
        status_print("Transcripts by company:")
        for ticker, count in company_counts.items():
            status_print(f"  {ticker}: {count} transcripts")

        # Count how many from each source
        source_counts = {}
        for t in all_transcripts:
            source = t['source']
            source_counts[source] = source_counts.get(source, 0) + 1

        status_print("Transcripts by source:")
        for source, count in source_counts.items():
            status_print(f"  {source}: {count} transcripts")
    else:
        status_print("❌ No transcripts were collected.")

    # Save all our metadata
    status_print("\n📝 Updating transcript metadata file...")
    update_metadata(
        all_transcripts, 
        metadata_file=TRANSCRIPT_METADATA,
        base_dir=BASE_DIR
    )

    # Show how long it all took
    end_time = datetime.now()
    duration = end_time - start_time
    hours, remainder = divmod(duration.seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    status_print(f"\n✅ TRANSCRIPT COLLECTION COMPLETE!")
    status_print(f"Started: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    status_print(f"Finished: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
    status_print(f"Duration: {hours}h {minutes}m {seconds}s")
    status_print(f"Metadata file: {TRANSCRIPT_METADATA}")

if __name__ == "__main__":
    main()


[16:12:48] 🚀 STARTING EARNINGS CALL TRANSCRIPT COLLECTOR 🚀
[16:12:48] Output directory: C:\Users\luke3\Documents\GitHub\Earnings Call Analyzer\data\transcripts
[16:12:48] Will process 24 companies
[16:12:48] 1. AAPL - Apple Inc.
[16:12:48] 2. MSFT - Microsoft Corporation
[16:12:48] 3. GOOGL - Alphabet Inc.
[16:12:48] 4. AMZN - Amazon.com Inc.
[16:12:48] 5. META - Meta Platforms Inc.
[16:12:48] 6. TSLA - Tesla Inc.
[16:12:48] 7. NVDA - NVIDIA Corporation
[16:12:48] 8. NFLX - Netflix Inc.
[16:12:48] 9. JPM - JPMorgan Chase & Co.
[16:12:48] 10. BAC - Bank of America Corporation
[16:12:48] 11. WFC - Wells Fargo & Company
[16:12:48] 12. GS - Goldman Sachs Group Inc.
[16:12:48] 13. V - Visa Inc.
[16:12:48] 14. WMT - Walmart Inc.
[16:12:48] 15. HD - Home Depot Inc.
[16:12:48] 16. PG - Procter & Gamble Company
[16:12:48] 17. KO - The Coca-Cola Company
[16:12:48] 18. PEP - PepsiCo Inc.
[16:12:48] 19. JNJ - Johnson & Johnson
[16:12:48] 20. PFE - Pfizer Inc.
[16:12:48] 21. UNH - UnitedHealth Grou

In [5]:
# Testing

# For faster testing (uncomment these)
#num_companies_to_process = 2  # Just process 2 companies  
#DELAY_BETWEEN_COMPANIES = 30  # Only wait 30 seconds